In [7]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
print("Python executable:", sys.executable)
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.models import Model

# Suppress verbose logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print("🚀 Num GPUs Available:", len(gpus))
if gpus:
    print("✅ GPU is available. Training will use GPU.")
else:
    print("⚠️ No GPU found. Training will use CPU (may be slower).")

# Set image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Only rescaling for validation data
val_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    r'D:\2D_to_3D\dataset\train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'  # two classes: TB and Normal
)

# Load validation data
val_generator = val_datagen.flow_from_directory(
    r'D:\2D_to_3D\dataset\val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load the pre-trained base model
base_model = MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
base_model.trainable = False  # Freeze base model

# Custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

# Combine base and custom head
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

# Save the model
model.save("tb_model.h5")
print("✅ Model saved as tb_model.h5")


: 

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Create a validation generator without shuffling for evaluation
val_eval_generator = val_datagen.flow_from_directory(
    r'D:\2D_to_3D\dataset\val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Predict on validation set
val_predictions = model.predict(val_eval_generator)
val_predictions_classes = np.argmax(val_predictions, axis=1)
val_true_classes = val_eval_generator.classes

# Get class names
target_names = list(val_eval_generator.class_indices.keys())

# Print classification report
print(classification_report(val_true_classes, val_predictions_classes, target_names=target_names))